# Chapter 7 — Enterprise RAG: Agentic Routing, Semantic Caching, and Query Rewriting

Companion code for **Chapter 7** of *Build an Advanced RAG Application (From Scratch)*.

The plain RAG of Chapter 6 starts to crack in enterprise settings — multiple knowledge bases, paraphrased questions hitting the LLM 100 times an hour, vague or compound queries that no single search nails. Chapter 7 fixes all three with three pillars:

| Pillar | What it does |
|--------|-------------|
| **Agentic Router** | Classifies each query and dispatches to the right knowledge base (or web) | 
| **Semantic Cache** | Reuses prior answers for paraphrases of the same question | 
| **Query Rewriter / Decomposer** | Polishes vague queries; splits compound ones into atomic sub-queries | 
| **Combined pipeline** | All three plus a time-sensitivity bypass | 


# 1. Setup

Add your keys in one of these ways:
- Colab Secrets: `OPENAI_API_KEY`, optionally `SERPAPI_KEY`, `QDRANT_URL`, `QDRANT_API_KEY`
- Environment variables in your runtime

The notebook downloads the Chapter 7 source documents directly from the repo


In [ ]:
from IPython.display import HTML, display

def set_css(*args, **kwargs):
    display(HTML("<style>pre { white-space: pre-wrap; }</style>"))

get_ipython().events.register('pre_run_cell', set_css)


In [ ]:
# Install necessary libraries
!pip install openai 
!pip install qdrant-client 
!pip install sentence-transformers 
!pip install faiss-cpu 
!pip install pymupdf 
!pip install python-dotenv 
!pip install nest_asyncio 
!pip install requests 
!pip install transformers 
!pip install einops


In [7]:
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def set_env_from_colab_secret(name: str, *, required: bool = False) -> str | None:
    if os.getenv(name):
        return os.getenv(name)
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
            return value
    if required:
        raise RuntimeError(f"Missing required key: {name}")
    return None


set_env_from_colab_secret("OPENAI_API_KEY", required=True)
set_env_from_colab_secret("SERPAPI_KEY")
set_env_from_colab_secret("QDRANT_URL")
set_env_from_colab_secret("QDRANT_API_KEY")

print("OPENAI_API_KEY loaded.")
print("SERPAPI_KEY loaded?", bool(os.getenv("SERPAPI_KEY")))
print("Using remote Qdrant?", bool(os.getenv("QDRANT_URL")))


OPENAI_API_KEY loaded.
SERPAPI_KEY loaded? False
Using remote Qdrant? False


In [ ]:
import asyncio
import json
import re
import time
from functools import lru_cache
from pathlib import Path
from uuid import uuid4

import faiss
import fitz
import nest_asyncio
import numpy as np
import requests
from openai import OpenAI, OpenAIError
from qdrant_client import AsyncQdrantClient, models
from sentence_transformers import SentenceTransformer

nest_asyncio.apply()

DATA_DIR = Path("chapter_7_data")
DATA_DIR.mkdir(exist_ok=True)

REPO_BASE = "https://raw.githubusercontent.com/hamzafarooq/advanced-rag-from-scratch/main/chapter_07_enterprise_rag/data"
DATA_FILES = {
    "openai_agents_guide.pdf": f"{REPO_BASE}/openai_agents_guide.pdf",
    "lyft_10k_2020.pdf": f"{REPO_BASE}/lyft_10k_2020.pdf",
    "lyft_10k_2021.pdf": f"{REPO_BASE}/lyft_10k_2021.pdf",
    "lyft_10k_2022.pdf": f"{REPO_BASE}/lyft_10k_2022.pdf",
    "uber_10k_2021.htm": f"{REPO_BASE}/uber_10k_2021.htm",
}


# 2. Download the chapter data

This pulls the same five source documents used by Chapter 7 into the notebook runtime.


In [9]:
# Download the chapter source files if they are not present locally.

def download_chapter_data(force: bool = False) -> None:
    for filename, url in DATA_FILES.items():
        target = DATA_DIR / filename
        if target.exists() and not force:
            print(f"exists: {filename}")
            continue
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        target.write_bytes(response.content)
        print(f"downloaded: {filename}")


download_chapter_data()


downloaded: openai_agents_guide.pdf
downloaded: lyft_10k_2020.pdf
downloaded: lyft_10k_2021.pdf
downloaded: lyft_10k_2022.pdf
downloaded: uber_10k_2021.htm


# 3. Core implementation

Everything below is inlined.



In [10]:
COLLECTIONS = {
    "OPENAI_QUERY": "opnai_data",
    "10K_DOCUMENT_QUERY": "10k_data",
}

EMBEDDING_DIM = 768
DEFAULT_SEPARATORS = ["\n\n", "\n", " ", ".", ",", "，", "、", "．", "。"]
TIME_SENSITIVE_KEYWORDS = [
    "today", "tonight", "now", "currently", "current",
    "latest", "recent", "recently", "right now",
    "at the moment", "at present", "as of now",
    "this week", "this month", "this year",
    "this quarter", "this season", "this morning",
    "this afternoon", "this evening", "this weekend",
    "yesterday", "tomorrow", "last week", "last month",
    "last year", "upcoming", "live", "breaking",
    "just happened", "what time", "what day", "what date",
    "happening now", "events today", "news today",
    "news this week", "stock price", "share price",
    "weather", "forecast", "temperature",
    "real-time", "realtime", "schedule today",
    "outage", "down right now",
]


def get_openai_client() -> OpenAI:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY is missing")
    return OpenAI(api_key=api_key)


def get_qdrant_async_client() -> AsyncQdrantClient:
    url = os.getenv("QDRANT_URL", ":memory:")
    api_key = os.getenv("QDRANT_API_KEY")
    if url == ":memory:" or not url:
        return AsyncQdrantClient(":memory:")
    return AsyncQdrantClient(url=url, api_key=api_key)


@lru_cache(maxsize=1)
def _get_encoder() -> SentenceTransformer:
    return SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)


def get_text_embedding(text: str) -> list[float]:
    vector = _get_encoder().encode([text], normalize_embeddings=True).astype("float32")[0]
    return vector.tolist()


def simple_recursive_split(
    doc: dict,
    chunk_size: int = 4000,
    chunk_overlap: int = 200,
    separators: list[str] | None = None,
) -> list[dict]:
    text = doc["page_content"]
    metadata = doc["metadata"]
    seps = separators or DEFAULT_SEPARATORS

    def split(t: str, sep_idx: int = 0) -> list[str]:
        if len(t) <= chunk_size:
            return [t]
        for j in range(sep_idx, len(seps)):
            sep = seps[j]
            if sep and sep in t:
                parts = t.split(sep)
                chunks, current = [], ""
                for part in parts:
                    part = part + sep
                    if len(current) + len(part) <= chunk_size:
                        current += part
                    else:
                        if current:
                            chunks.append(current)
                        current = part
                if current:
                    chunks.append(current)
                final = []
                for c in chunks:
                    final.extend(split(c, j + 1) if len(c) > chunk_size else [c])
                return final
        step = max(1, chunk_size - chunk_overlap)
        return [t[i : i + chunk_size] for i in range(0, len(t), step)]

    pieces = split(text)
    if chunk_overlap and len(pieces) > 1:
        overlapped = [pieces[0]]
        for i in range(1, len(pieces)):
            tail = pieces[i - 1][-chunk_overlap:]
            overlapped.append(tail + pieces[i])
        pieces = overlapped

    return [{"page_content": p, "metadata": metadata} for p in pieces]


def is_time_sensitive(question: str) -> bool:
    q = question.lower()
    return any(kw in q for kw in TIME_SENSITIVE_KEYWORDS)


class SemanticCaching:
    def __init__(
        self,
        json_file: str = "cache.json",
        threshold: float = 0.2,
        embedding_dim: int = 768,
        encoder_name: str = "nomic-ai/nomic-embed-text-v1.5",
        clear_on_init: bool = False,
    ) -> None:
        self.embedding_dim = embedding_dim
        self.json_file = Path(json_file)
        self.euclidean_threshold = threshold
        self.index = faiss.IndexFlatL2(self.embedding_dim)
        self.encoder = SentenceTransformer(encoder_name, trust_remote_code=True)
        self.cache: dict = {"questions": [], "embeddings": [], "response_text": []}

        if clear_on_init:
            self.clear_cache()
        else:
            self.load_cache()

    def load_cache(self) -> None:
        try:
            with self.json_file.open("r", encoding="utf-8") as f:
                self.cache = json.load(f)
            if self.cache["embeddings"]:
                vectors = np.array(self.cache["embeddings"], dtype=np.float32)
                self.index.add(vectors)
        except FileNotFoundError:
            self.cache = {"questions": [], "embeddings": [], "response_text": []}

    def check_cache(self, question: str):
        embedding = self.encoder.encode([question], normalize_embeddings=True).astype("float32")
        if self.index.ntotal == 0:
            return False, None, embedding, None, None

        distances, indices = self.index.search(embedding, 1)
        idx = int(indices[0][0])
        dist = float(distances[0][0])
        if idx != -1 and dist <= self.euclidean_threshold:
            return True, self.cache["response_text"][idx], embedding, 1.0 - dist, idx
        return False, None, embedding, None, None

    def add_to_cache(self, question: str, answer: str, embedding) -> None:
        self.cache["questions"].append(question)
        self.cache["embeddings"].append(embedding[0].tolist())
        self.cache["response_text"].append(answer)
        self.index.add(embedding)
        self.save_cache()

    def save_cache(self) -> None:
        with self.json_file.open("w", encoding="utf-8") as f:
            json.dump(self.cache, f)

    def clear_cache(self) -> None:
        self.cache = {"questions": [], "embeddings": [], "response_text": []}
        self.index = faiss.IndexFlatL2(self.embedding_dim)
        self.save_cache()


In [11]:
# Classify each question before choosing retrieval or live search.

def route_query(user_query: str, *, llm_model: str = "gpt-4o") -> dict:
    client = get_openai_client()
    router_system_prompt = f"""As a professional query router, classify user input into one of three categories:

1. "OPENAI_QUERY": Questions about OpenAI documentation -- agents, tools, APIs,
   models, embeddings, guardrails, the Responses API, or Assistants API.
2. "10K_DOCUMENT_QUERY": Questions about company financials, 10-K annual reports,
   Uber or Lyft revenue, operating costs, or filing disclosures.
3. "WEB_SEARCH": Everything else -- general knowledge, technology trends,
   comparisons, or anything not in the internal document collections.

Always respond in this exact JSON format:
{{
    "action": "OPENAI_QUERY" or "10K_DOCUMENT_QUERY" or "WEB_SEARCH",
    "reason": "one sentence justification for the routing decision",
    "answer": "AT MOST 5 words if trivially obvious, else leave empty"
}}

User: {user_query}
"""
    try:
        response = client.chat.completions.create(
            model=llm_model,
            messages=[{"role": "system", "content": router_system_prompt}],
        )
        text = response.choices[0].message.content
        match = re.search(r"\{.*\}", text, re.DOTALL)
        return json.loads(match.group())
    except (OpenAIError, json.JSONDecodeError, AttributeError) as err:
        return {"action": "WEB_SEARCH", "reason": f"Routing error: {err}", "answer": ""}


def rag_formatted_response(user_query: str, context: list, *, llm_model: str = "gpt-4o") -> str:
    client = get_openai_client()
    rag_prompt = f"""Based on the given context, answer the user query: {user_query}
Context: {context}
Use numbered citations [1][2][3] referencing the context chunks.
Begin directly with the answer.
"""
    response = client.chat.completions.create(
        model=llm_model,
        messages=[{"role": "system", "content": rag_prompt}],
    )
    return response.choices[0].message.content


async def retrieve_and_respond(user_query: str, action: str, qdrant: AsyncQdrantClient, *, k: int = 3) -> str:
    query_embedding = get_text_embedding(user_query)
    hits = await qdrant.query_points(
        collection_name=COLLECTIONS[action],
        query=query_embedding,
        limit=k,
    )
    contents = [point.payload["content"] for point in hits.points]
    return rag_formatted_response(user_query, contents)


def search_web(user_query: str, *, max_results: int = 5) -> list[str]:
    api_key = os.getenv("SERPAPI_KEY")
    if not api_key:
        return [
            f"[stub] Web search disabled: set SERPAPI_KEY to enable. Query was: {user_query}"
        ]
    try:
        params = {"q": user_query, "api_key": api_key, "engine": "google"}
        response = requests.get("https://serpapi.com/search.json", params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        organic = data.get("organic_results", [])[:max_results]
        return [item.get("snippet", "") for item in organic if item.get("snippet")]
    except Exception as err:
        return [f"[error] SerpAPI failed ({err}). Query was: {user_query}"]


async def handle_query(user_query: str, qdrant: AsyncQdrantClient) -> str:
    route = route_query(user_query)
    print(f"Route: {route['action']}\nReason: {route['reason']}")

    if route.get("answer"):
        return route["answer"]

    if route["action"] == "WEB_SEARCH":
        return rag_formatted_response(user_query, search_web(user_query))

    return await retrieve_and_respond(user_query, route["action"], qdrant)


def rewrite_query(
    user_query: str,
    conversation_history: list[tuple[str, str]] | None = None,
    *,
    llm_model: str = "gpt-4o",
) -> str:
    history_context = ""
    if conversation_history:
        history_context = "\n".join(
            f"Q: {q}\nA: {a[:200]}..." for q, a in conversation_history[-3:]
        )

    rewrite_prompt = f"""You are a search query optimizer. Rewrite the user's query to make it more
precise and retrieval-friendly.

Rules:
1. Expand abbreviations ("Q3" -> "third quarter", "rev" -> "revenue")
2. Replace vague references with specific terms using conversation history
3. Add relevant domain context (year, company name) when clearly implied
4. Do NOT add constraints the user did not express
5. Return ONLY the rewritten query, no explanation

Conversation history:
{history_context if history_context else "None"}

Original query: {user_query}
Rewritten query:
"""
    client = get_openai_client()
    response = client.chat.completions.create(
        model=llm_model,
        messages=[{"role": "user", "content": rewrite_prompt}],
        max_tokens=200,
        temperature=0,
    )
    return response.choices[0].message.content.strip()


def decompose_query(user_query: str, *, llm_model: str = "gpt-4o") -> list[str]:
    decompose_prompt = f"""Analyze the following query and determine if it contains multiple distinct
information needs. If it does, break it into 2-4 focused atomic sub-queries.
If it is already a single focused question, return it unchanged.

Rules:
- Each sub-query must be independently answerable
- Sub-queries should not overlap or repeat each other
- Preserve specific entities (company names, time periods)
- Return ONLY a JSON array of strings

Query: {user_query}
"""
    client = get_openai_client()
    response = client.chat.completions.create(
        model=llm_model,
        messages=[{"role": "user", "content": decompose_prompt}],
        temperature=0,
    )
    try:
        text = response.choices[0].message.content.strip()
        match = re.search(r"\[.*\]", text, re.DOTALL)
        return json.loads(match.group())
    except (json.JSONDecodeError, AttributeError):
        return [user_query]


In [12]:
# Load and chunk the OpenAI docs and 10-K source files for indexing.

OPENAI_FILES = [DATA_DIR / "openai_agents_guide.pdf"]
TENK_FILES = [
    DATA_DIR / "lyft_10k_2020.pdf",
    DATA_DIR / "lyft_10k_2021.pdf",
    DATA_DIR / "lyft_10k_2022.pdf",
    DATA_DIR / "uber_10k_2021.htm",
]


def read_pdf(path: Path) -> str:
    doc = fitz.open(path)
    return "".join(page.get_text() for page in doc)


def read_htm(path: Path) -> str:
    raw = path.read_text(encoding="utf-8", errors="ignore")
    text = re.sub(r"<[^>]+>", " ", raw)
    text = re.sub(r"&nbsp;", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def load_document(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return read_pdf(path)
    if suffix in (".htm", ".html"):
        return read_htm(path)
    raise ValueError(f"Unsupported file type: {path}")


def chunk_documents(paths: list[Path], *, chunk_size: int = 2048, chunk_overlap: int = 50):
    chunks: list[dict] = []
    for path in paths:
        if not path.exists():
            print(f"skipping (missing): {path.name}")
            continue
        text = load_document(path)
        if not text.strip():
            print(f"skipping (empty): {path.name}")
            continue
        doc = {"page_content": text, "metadata": {"document_info": str(path)}}
        new_chunks = simple_recursive_split(doc, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        for c in new_chunks:
            c["metadata"]["uuid"] = str(uuid4())
        chunks.extend(new_chunks)
        print(f"{path.name:<32} -> {len(new_chunks):>4} chunks")
    return chunks


async def build_collection(qdrant: AsyncQdrantClient, collection_name: str, chunks: list[dict]) -> None:
    if await qdrant.collection_exists(collection_name):
        await qdrant.delete_collection(collection_name=collection_name)
    await qdrant.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(size=EMBEDDING_DIM, distance=models.Distance.COSINE),
    )

    encoder = _get_encoder()
    texts = [c["page_content"] for c in chunks]
    print(f"embedding {len(texts)} chunks for '{collection_name}'...")
    vectors = encoder.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype("float32")

    points = [
        models.PointStruct(
            id=chunks[i]["metadata"]["uuid"],
            vector=vectors[i].tolist(),
            payload={"metadata": chunks[i]["metadata"], "content": chunks[i]["page_content"]},
        )
        for i in range(len(chunks))
    ]
    await qdrant.upsert(collection_name=collection_name, wait=True, points=points)
    print(f"upserted {len(points)} points into '{collection_name}'")


async def ingest_all(qdrant: AsyncQdrantClient) -> None:
    print("Chunking OpenAI agents guide...")
    openai_chunks = chunk_documents(OPENAI_FILES)
    print("\nChunking 10-K filings...")
    tenk_chunks = chunk_documents(TENK_FILES)

    print("\nBuilding Qdrant collections...")
    await build_collection(qdrant, COLLECTIONS["OPENAI_QUERY"], openai_chunks)
    await build_collection(qdrant, COLLECTIONS["10K_DOCUMENT_QUERY"], tenk_chunks)
    print("\nDone.")


async def enterprise_rag_pipeline(
    user_query: str,
    cache: SemanticCaching,
    qdrant: AsyncQdrantClient,
    *,
    conversation_history: list[tuple[str, str]] | None = None,
) -> dict:
    result = {
        "query": user_query,
        "rewritten_query": None,
        "sub_queries": None,
        "route": None,
        "reason": None,
        "cache_hit": False,
        "time_sensitive": False,
        "answer": None,
    }

    hit, cached_answer, embedding, _sim, _idx = cache.check_cache(user_query)
    if hit:
        result["cache_hit"] = True
        result["answer"] = cached_answer
        return result

    if is_time_sensitive(user_query):
        result["time_sensitive"] = True
        result["answer"] = rag_formatted_response(user_query, search_web(user_query))
        return result

    route = route_query(user_query)
    result["route"] = route["action"]
    result["reason"] = route["reason"]

    if route.get("answer"):
        result["answer"] = route["answer"]
        cache.add_to_cache(user_query, result["answer"], embedding)
        return result

    rewritten = rewrite_query(user_query, conversation_history)
    result["rewritten_query"] = rewritten

    sub_queries = decompose_query(rewritten)
    result["sub_queries"] = sub_queries

    action = route["action"]
    all_context: list[str] = []

    if action == "WEB_SEARCH":
        for sq in sub_queries:
            all_context.extend(search_web(sq))
    else:
        collection = COLLECTIONS[action]
        for sq in sub_queries:
            sq_embedding = get_text_embedding(sq)
            hits = await qdrant.query_points(collection_name=collection, query=sq_embedding, limit=3)
            all_context.extend(p.payload["content"] for p in hits.points)

    result["answer"] = rag_formatted_response(user_query, all_context)
    cache.add_to_cache(user_query, result["answer"], embedding)
    return result


# 4. Ingest the corpora into Qdrant

The first run embeds the source documents and loads them into two Qdrant collections.


In [13]:
# Push all prepared document chunks into Qdrant.

qdrant = get_qdrant_async_client()
asyncio.run(ingest_all(qdrant))


Chunking OpenAI agents guide...
openai_agents_guide.pdf          ->   20 chunks

Chunking 10-K filings...
lyft_10k_2020.pdf                ->  307 chunks
lyft_10k_2021.pdf                ->  437 chunks
lyft_10k_2022.pdf                ->  377 chunks
uber_10k_2021.htm                ->   57 chunks

Building Qdrant collections...


<All keys matched successfully>


embedding 20 chunks for 'opnai_data'...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

upserted 20 points into 'opnai_data'
embedding 1178 chunks for '10k_data'...


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

upserted 1178 points into '10k_data'

Done.


## Pillar 1: Agentic routing


In [14]:
for q in [
    "How do I create an OpenAI assistant with file search?",
    "What was Uber's gross bookings in Q3 2021?",
    "What are the most popular open-source LLMs in 2025?",
    "What does SDK stand for?",
]:
    print(f"Q: {q}")
    decision = route_query(q)
    print(f"  -> {decision['action']}  ({decision['reason']})")
    if decision["answer"]:
        print(f"     direct answer: {decision['answer']}")
    print()


Q: How do I create an OpenAI assistant with file search?
  -> OPENAI_QUERY  (The question is about creating an OpenAI assistant, likely involving API or agent use.)

Q: What was Uber's gross bookings in Q3 2021?
  -> 10K_DOCUMENT_QUERY  (Inquiry about Uber's financial details related to a specific quarter.)

Q: What are the most popular open-source LLMs in 2025?
  -> WEB_SEARCH  (The question involves future predictions beyond the training data.)

Q: What does SDK stand for?
  -> WEB_SEARCH  (The query asks for general knowledge.)
     direct answer: Software Development Kit



In [15]:
print(asyncio.run(handle_query(
    "How do I create an OpenAI assistant with file search?",
    qdrant,
)))


Route: OPENAI_QUERY
Reason: Question about OpenAI's assistant creation using files.
To create an OpenAI assistant with file search capabilities, you can start by building an intelligent system that integrates natural language processing with search functionalities. Here are some steps you may follow:

1. **Select an AI model**: Use a model like OpenAI's GPT, which is designed for understanding and generating human-like text.

2. **Implement file search functionality**: Incorporate a search algorithm to process and index files that your assistant needs to access. This could involve using a library to read the content of various file types and then creating an index to quickly search through the text data.

3. **Create a User Interface**: Develop an interface where users can interact with the assistant. This could be a chat window where users input queries related to file content they want to find.

4. **Integrate AI with search**: Combine the natural language understanding capabilities 

In [16]:
print(asyncio.run(handle_query(
    "What was Uber's revenue in 2021?",
    qdrant,
)))


Route: 10K_DOCUMENT_QUERY
Reason: The query involves company financials.
Uber's revenue in 2021 was $17.45 billion.


## Pillar 2: Semantic cache


In [17]:
for q in ["What is OpenAI?", "What's the weather today?", "Latest GPT news this week"]:
    print(f"{is_time_sensitive(q):>5}  {q}")


    0  What is OpenAI?
    1  What's the weather today?
    1  Latest GPT news this week


In [18]:
# Show how semantic caching turns a repeated question into a fast hit.

cache = SemanticCaching(clear_on_init=True)

q1 = "What was Uber's revenue in 2021?"
hit, _, embedding, _, _ = cache.check_cache(q1)
assert not hit
answer = asyncio.run(handle_query(q1, qdrant))
cache.add_to_cache(q1, answer, embedding)
print("First call (miss):\n", answer[:240], "...\n")

q2 = "How much did Uber earn in fiscal year 2021?"
hit, cached, _, sim, _ = cache.check_cache(q2)
print(f"Paraphrase hit? {hit}  (sim={sim:.3f})")
print("Cached answer:\n", cached[:240], "...")


<All keys matched successfully>


Route: 10K_DOCUMENT_QUERY
Reason: The question pertains to company financials and 10-K reports.
First call (miss):
 I apologize, but the information regarding Uber's revenue in 2021 is not available in the provided context. Based on external data, Uber's revenue in 2021 was approximately $17.5 billion. ...

Paraphrase hit? True  (sim=0.838)
Cached answer:
 I apologize, but the information regarding Uber's revenue in 2021 is not available in the provided context. Based on external data, Uber's revenue in 2021 was approximately $17.5 billion. ...


## Pillar 3: Query rewriting and decomposition


In [19]:
# Rewrite a follow-up question so it can stand on its own.

history = [
    ("Tell me about Uber's 2021 financials", "Uber's 2021 revenue was $17.5B, up 57% YoY..."),
]
print(rewrite_query("How does that compare to Lyft?", conversation_history=history))


How does Uber's 2021 revenue compare to Lyft's 2021 revenue?


In [20]:
# Split a compound question into smaller retrieval-friendly parts.

compound = "What was Uber's revenue in 2021 and how does their gross bookings growth compare to Lyft's?"
for sq in decompose_query(compound):
    print(" -", sq)


 - What was Uber's revenue in 2021?
 - How does Uber's gross bookings growth compare to Lyft's?


# 5. Combined enterprise pipeline


In [21]:
# Run a small end-to-end test suite across the enterprise RAG features.

cache = SemanticCaching(clear_on_init=True)

test_queries = [
    ("Cache miss, 10-K route", "What was Uber's revenue in 2021?"),
    ("Cache hit on paraphrase", "How much money did Uber make in 2021?"),
    ("Time-sensitive bypass", "What are the latest AI model releases this week?"),
    ("Compound, needs decomposition", "Q3 rev breakdown for the two ride-share companies"),
    ("OpenAI docs route", "How do I create an assistant with file search?"),
]

for label, q in test_queries:
    print("=" * 70)
    print(f"[{label}]  {q}")
    print("=" * 70)
    result = asyncio.run(enterprise_rag_pipeline(q, cache, qdrant))
    print(f"cache_hit       : {result['cache_hit']}")
    print(f"time_sensitive  : {result['time_sensitive']}")
    print(f"route           : {result['route']}")
    print(f"reason          : {result['reason']}")
    print(f"rewritten_query : {result['rewritten_query']}")
    print(f"sub_queries     : {result['sub_queries']}")
    print(f"answer          : {(result['answer'] or '')[:240]}...")
    print()


<All keys matched successfully>


[Cache miss, 10-K route]  What was Uber's revenue in 2021?
cache_hit       : False
time_sensitive  : False
route           : 10K_DOCUMENT_QUERY
reason          : The question pertains to Uber's financials.
rewritten_query : What was Uber's revenue in the year 2021?
sub_queries     : ["What was Uber's revenue in the year 2021?"]
answer          : Uber's revenue in 2021 was not provided in the given context. The context only contains information related to Lyft, Inc. and its financial certifications. Therefore, I cannot provide Uber's 2021 revenue based on this information....

[Cache hit on paraphrase]  How much money did Uber make in 2021?
cache_hit       : True
time_sensitive  : False
route           : None
reason          : None
rewritten_query : None
sub_queries     : None
answer          : Uber's revenue in 2021 was not provided in the given context. The context only contains information related to Lyft, Inc. and its financial certifications. Therefore, I cannot provide Uber's 2021

# 6. Cache speedup check


In [22]:
# Compare latency for a cache miss versus a semantic cache hit.

cache = SemanticCaching(clear_on_init=True)

t0 = time.time()
asyncio.run(enterprise_rag_pipeline("What was Uber's revenue in 2021?", cache, qdrant))
miss_time = time.time() - t0

t0 = time.time()
asyncio.run(enterprise_rag_pipeline("How much did Uber earn in fiscal year 2021?", cache, qdrant))
hit_time = time.time() - t0

print(f"Cache MISS: {miss_time:.2f}s")
print(f"Cache HIT:  {hit_time:.3f}s")
print(f"Speedup:    {miss_time / max(hit_time, 1e-3):.0f}x")


<All keys matched successfully>


Cache MISS: 5.97s
Cache HIT:  0.054s
Speedup:    110x


## What's next

Chapter 8 takes the pieces from Chapters 6 and 7 — retrieval, prompting, routing, caching, rewriting — and packages them for production: deployment, observability, guardrails, and full agentic orchestration where autonomous agents drive the pipeline.